# FIT TRADIZIONALE con multi-start Monte Carlo


---

Fit di tutti gli spettri usando le classi [PoissonNoise](/ml_odmr/PoissonNoise.py) e [FitMonteCarlo](/ml_odmr/FitMonteCarlo.py).

In [ ]:
list_contrast = [0.8, 0.5, 0.2, 0.1, 0.08, 0.05, 0.02, 0.01]
list_Nph = [3e5, 6e5, 12e5, 24e5, 48e5, 60e5]
list_mc_trials = [1, 5, 10, 20, 40]

In [ ]:
import os
import time
from datetime import datetime
import h5py
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from ml_odmr.FitMonteCarlo import FitMonteCarlo
from ml_odmr.PoissonNoise import PoissonNoise

# =========================================================================
# 1. CONFIGURAZIONE DEL RUN E CARICAMENTO DATI
# =========================================================================
mat_file_path = "../data/10k/odmr_easyspin_raw_10000.mat" 

# Parametri fisici e statistici del test
n_photons_selected = 48.0e5      # Budget di fotoni fissato per questa campagna
contrast_selected = 0.08        # Contrasto ODMR fissato (8%)
monte_carlo_trials = 20         # Tentativi multi-start M per spettro
random_seed = 42                # Seed di base per la riproducibilità

# Nome del file di output generato automaticamente in base ai fotoni
output_csv_path = f"benchmark_mc{monte_carlo_trials}_Nph{int(n_photons_selected)}_cont{contrast_selected}.csv"

# Lettura del file .mat generato da EasySpin
with h5py.File(mat_file_path, "r") as f:
    # Ripristiniamo la forma [N_spettri, N_punti] con la trasposta .T
    spectra_data = np.array(f["spectra_data"]).T
    mw_freqs = np.array(f["mw_freqs"]).squeeze() 
    labels_c = np.array(f["labels_c"]).squeeze() 
    labels_s = np.array(f["labels_s"]).squeeze() 

f_ghz = np.asarray(mw_freqs, dtype=np.float64) 
N_spectra = len(spectra_data)

# Istanziazione delle classi
noise_gen = PoissonNoise(seed=random_seed)
fitter = FitMonteCarlo(M=monte_carlo_trials)


# =========================================================================
# 2. DEFINIZIONE DEL WORKER PER IL SINGOLO SPETTRO
# =========================================================================
def process_single_spectrum(idx: int) -> dict:
    """
    Applica il rumore allo spettro idx-esimo, esegue il fit Monte Carlo
    e restituisce un dizionario con i risultati e gli errori rispetto alla ground truth.
    """
    y_raw = spectra_data[idx]
    
    # 1. Generazione segnale rumoroso
    ideal_sig = noise_gen.ideal_signal(y_raw, contrast=contrast_selected)
    noisy_sig = noise_gen.add_noise(
        ideal_sig, 
        n_tot=n_photons_selected, 
        seed_modifier=idx
    )
    
    # 2. Esecuzione del fit (restituisce un oggetto FitResult)
    res = fitter.fit_single(f_ghz, noisy_sig)
    
    # 3. Ground truth di riferimento
    c_true = float(labels_c[idx])
    s_true = float(labels_s[idx])
    
    # 4. Calcolo errori assoluti nello spazio normalizzato [0, 1][cite: 2]
    err_c = abs(res.c_norm - c_true) if res.converged else np.nan
    err_s = abs(res.s_norm - s_true) if res.converged else np.nan
    
    return {
        "spectrum_id": idx,
        "converged": res.converged,
        "c_true_norm": c_true,
        "s_true_norm": s_true,
        "c_fit_norm": res.c_norm,
        "s_fit_norm": res.s_norm,
        "err_c_norm": err_c,
        "err_s_norm": err_s,
        "c_fit_mhz": res.c_mhz,
        "s_fit_mhz": res.s_mhz,
        "chi2_red": res.chi2_red,
        "fit_time_sec": res.duration_sec
    }


# =========================================================================
# 3. ESECUZIONE PARALLELA SU TUTTI GLI SPETTRI
# =========================================================================
print(f"Avvio Fit Monte Carlo su {N_spectra} spettri (M={monte_carlo_trials}, N_photons={n_photons_selected:.1e})...")
t_start_wall = time.time()

# n_jobs=-1 utilizza tutti i core disponibili del processore M1
results_dicts = Parallel(n_jobs=-1, verbose=5)(
    delayed(process_single_spectrum)(i) for i in range(N_spectra)
)

total_wall_time = time.time() - t_start_wall
print(f"\nElaborazione completata in {total_wall_time:.2f} secondi!")


# =========================================================================
# 4. CREAZIONE DATAFRAME E SCRITTURA DEL FILE CSV CON HEADER
# =========================================================================
df_results = pd.DataFrame(results_dicts)

# Statistiche globali da inserire nell'header del file[cite: 2]
n_converged = int(df_results["converged"].sum())
success_rate = (n_converged / N_spectra) * 100.0
mae_c_tot = df_results["err_c_norm"].mean()
mae_s_tot = df_results["err_s_norm"].mean()
mean_fit_time = df_results["fit_time_sec"].mean()

header_metadata = (
    f"# ODMR Monte Carlo Fit Benchmark\n"
    f"# Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    f"# Source File: {os.path.basename(mat_file_path)}\n"
    f"# N_spectra: {N_spectra}\n"
    f"# N_photons: {n_photons_selected:.2e}\n"
    f"# Contrast: {contrast_selected}\n"
    f"# MC_Trials_M: {monte_carlo_trials}\n"
    f"# Base_Seed: {random_seed}\n"
    f"# Converged_Spectra: {n_converged}/{N_spectra} ({success_rate:.2f}%)\n"
    f"# Mean_MAE_c_norm: {mae_c_tot:.6f}\n"
    f"# Mean_MAE_s_norm: {mae_s_tot:.6f}\n"
    f"# Mean_Fit_Time_Per_Spectrum_s: {mean_fit_time:.4f}\n"
    f"# Total_Wall_Time_s: {total_wall_time:.2f}\n"
)

# Scrittura su disco: scriviamo prima i metadati commentati con '#', poi la tabella CSV
with open(output_csv_path, mode="w", encoding="utf-8") as f:
    f.write(header_metadata)
    df_results.to_csv(f, index=False, float_format="%.6f")

print(f"File salvato correttamente in: {output_csv_path}")
print(f"Tasso di convergenza: {success_rate:.2f}% | MAE(c): {mae_c_tot:.5f} | MAE(s): {mae_s_tot:.5f}")

Avvio Fit Monte Carlo su 10000 spettri (M=20, N_photons=4.8e+06)...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    9.8s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   22.2s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:   41.0s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 632 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 866 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 1136 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-1)]: Done 1442 tasks      | elapsed:  3.5min
[Parallel(n_jobs=-1)]: Done 1784 tasks      | elapsed:  4.2min
[Parallel(n_jobs=-1)]: Done 2162 tasks      | elapsed:  5.0min
[Parallel(n_jobs=-1)]: Done 2576 tasks      | elapsed:  5.9min
[Parallel(n_jobs=-1)]: Done 3026 tasks      | elapsed:  6.9min
[Parallel(n_jobs=-1)]: Done 3512 tasks      | elapsed:  7.9min
[Parallel(n_jobs=-1)]: Done 4034 tasks      | ela


Elaborazione completata in 1481.18 secondi!
File salvato correttamente in: benchmark_mc_fit_Nph4800000_cont0.08.csv
Tasso di convergenza: 100.00% | MAE(c): 0.00099 | MAE(s): 0.00136
